# Chicago Food Inspections + 311 Sanitation — Final Data Cleaning

This notebook keeps only the cleaning steps used in the final workflow and removes repeated exploration, testing, and intermediate versions.

## Cleaning Principles
1. **Do not impute values just to eliminate missing data.** Impute only when other fields provide strong evidence.
2. **Preserve original information whenever possible.** Helper columns are used during cleaning, and only final cleaned fields are kept for analysis.
3. **Validate important imputation rules before applying them.**


## 1. Setup and Load Data

The original datasets are left unchanged. All cleaning is performed on `food_test` and `sanitation_test` copies.


In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.neighbors import KNeighborsClassifier, BallTree

food_path = "/Users/michael818/Desktop/BA780_dataCleaning/Food_Inspections_2010_2026.csv"
sanitation_path = "/Users/michael818/Desktop/BA780_dataCleaning/311_Sanitation_Combined_2010_2026.csv"

food = pd.read_csv(food_path)
sanitation = pd.read_csv(sanitation_path)

food_test = food.copy()
sanitation_test = sanitation.copy()

print("Food:", food_test.shape)
print("311:", sanitation_test.shape)

# Part A — Food Inspections

## 2. Facility Type

`Facility Type` has 5,339 missing values. We do not fill them with the overall mode because different establishments can have different facility types.

Imputation priority:
1. **License #** — strongest establishment-level identifier.
2. **DBA Name + Address** — used when the license number cannot provide a reliable value.

Only high-confidence historical matches are used.


In [ ]:
# Clean invalid license identifier
food_test["License_clean"] = food_test["License #"].replace(0, np.nan)

# Keep original values and record the source of every imputed value
food_test["Facility Type Imputed"] = food_test["Facility Type"]
food_test["Facility Type Imputation Method"] = np.where(
    food_test["Facility Type"].notna(),
    "Original",
    "Still Missing"
)

def mode_stats(df, group_cols, target):
    stats = (
        df.dropna(subset=group_cols + [target])
          .groupby(group_cols)[target]
          .agg(
              predicted=lambda x: x.mode().iloc[0],
              total_records="count",
              mode_count=lambda x: x.value_counts().iloc[0]
          )
    )
    stats["confidence"] = stats["mode_count"] / stats["total_records"]
    return stats

# ---------- 1) License # ----------
license_stats = mode_stats(
    food_test,
    ["License_clean"],
    "Facility Type"
)

license_prediction = food_test["License_clean"].map(license_stats["predicted"])
license_confidence = food_test["License_clean"].map(license_stats["confidence"])

mask = (
    food_test["Facility Type Imputed"].isna()
    & license_prediction.notna()
    & (license_confidence >= 0.90)
)

food_test.loc[mask, "Facility Type Imputed"] = license_prediction[mask]
food_test.loc[mask, "Facility Type Imputation Method"] = "License #"

# ---------- 2) DBA Name + Address ----------
dba_address_stats = mode_stats(
    food_test,
    ["DBA Name", "Address"],
    "Facility Type"
)

lookup_index = pd.MultiIndex.from_frame(food_test[["DBA Name", "Address"]])

dba_address_prediction = pd.Series(
    dba_address_stats["predicted"].reindex(lookup_index).to_numpy(),
    index=food_test.index
)
dba_address_confidence = pd.Series(
    dba_address_stats["confidence"].reindex(lookup_index).to_numpy(),
    index=food_test.index
)

mask = (
    food_test["Facility Type Imputed"].isna()
    & dba_address_prediction.notna()
    & (dba_address_confidence >= 0.90)
)

food_test.loc[mask, "Facility Type Imputed"] = dba_address_prediction[mask]
food_test.loc[mask, "Facility Type Imputation Method"] = "DBA Name + Address"

# ---------- 3) DBA Name ----------
dba_stats = mode_stats(
    food_test,
    ["DBA Name"],
    "Facility Type"
)

dba_prediction = food_test["DBA Name"].map(dba_stats["predicted"])
dba_confidence = food_test["DBA Name"].map(dba_stats["confidence"])

mask = (
    food_test["Facility Type Imputed"].isna()
    & dba_prediction.notna()
    & (dba_confidence >= 0.90)
)

food_test.loc[mask, "Facility Type Imputed"] = dba_prediction[mask]
food_test.loc[mask, "Facility Type Imputation Method"] = "DBA Name"

print("Facility Type imputation:")
print(food_test["Facility Type Imputation Method"].value_counts())
print("Remaining missing:", food_test["Facility Type Imputed"].isna().sum())

### Standardize Facility Type Labels

`Facility Type` contains differences in capitalization, spacing, and punctuation, such as `GAS STATION` vs. `gas station` and `ROOF TOP` vs. `ROOFTOP`.

We first create a lowercase, trimmed comparison key, then keep the most common original label within each standardized group. This reduces duplicate categories without inventing new facility types.


In [ ]:
# Basic formatting normalization
food_test["Facility Type Key"] = (
    food_test["Facility Type Imputed"]
    .str.strip()
    .str.lower()
    .str.replace(r"\s+", " ", regex=True)
)

canonical_basic = (
    food_test.dropna(subset=["Facility Type Imputed"])
    .groupby("Facility Type Key")["Facility Type Imputed"]
    .agg(lambda x: x.value_counts().index[0])
)

food_test["Facility Type Clean"] = (
    food_test["Facility Type Key"].map(canonical_basic)
)

# More aggressive comparison key for punctuation/spacing variants
food_test["Facility Compare"] = (
    food_test["Facility Type Clean"]
    .str.lower()
    .str.replace(r"[^a-z0-9]", "", regex=True)
)

candidate_groups = (
    food_test.dropna(subset=["Facility Type Clean"])
    .groupby("Facility Compare")["Facility Type Clean"]
    .unique()
)

candidate_groups = candidate_groups[candidate_groups.apply(len) > 1]

# These groups were manually reviewed earlier and found to represent
# formatting/naming variants rather than substantively different categories.
final_mapping = {}

for _, values in candidate_groups.items():
    counts = (
        food_test.loc[
            food_test["Facility Type Clean"].isin(values),
            "Facility Type Clean"
        ]
        .value_counts()
    )
    canonical = counts.index[0]
    for value in values:
        final_mapping[value] = canonical

food_test["Facility Type Final"] = (
    food_test["Facility Type Clean"].replace(final_mapping)
)

print("Original categories:", food_test["Facility Type"].nunique())
print("Final categories:", food_test["Facility Type Final"].nunique())
print("Remaining missing:", food_test["Facility Type Final"].isna().sum())

## 3. Risk

`Risk` has only a small number of missing values. `All` is **not treated as missing** because it appears in special inspection contexts such as `License`, `Not Ready`, `No Entry`, and `Out of Business`.

Missing Risk values are imputed only from high-confidence historical records, using License # first and DBA Name + Address second.


In [ ]:
normal_risk = [
    "Risk 1 (High)",
    "Risk 2 (Medium)",
    "Risk 3 (Low)"
]

food_test["Risk Imputed"] = food_test["Risk"]
food_test["Risk Imputation Method"] = np.where(
    food_test["Risk"].notna(),
    "Original",
    "Still Missing"
)

# ---------- 1) License # ----------
risk_license_stats = (
    food_test[
        food_test["Risk"].isin(normal_risk)
        & food_test["License_clean"].notna()
    ]
    .groupby("License_clean")["Risk"]
    .agg(
        predicted=lambda x: x.mode().iloc[0],
        total_records="count",
        mode_count=lambda x: x.value_counts().iloc[0]
    )
)
risk_license_stats["confidence"] = (
    risk_license_stats["mode_count"] / risk_license_stats["total_records"]
)

prediction = food_test["License_clean"].map(risk_license_stats["predicted"])
confidence = food_test["License_clean"].map(risk_license_stats["confidence"])

mask = (
    food_test["Risk Imputed"].isna()
    & prediction.notna()
    & (confidence >= 0.90)
)

food_test.loc[mask, "Risk Imputed"] = prediction[mask]
food_test.loc[mask, "Risk Imputation Method"] = "License #"

# ---------- 2) DBA Name + Address ----------
risk_dba_address_stats = (
    food_test[food_test["Risk"].isin(normal_risk)]
    .dropna(subset=["DBA Name", "Address"])
    .groupby(["DBA Name", "Address"])["Risk"]
    .agg(
        predicted=lambda x: x.mode().iloc[0],
        total_records="count",
        mode_count=lambda x: x.value_counts().iloc[0]
    )
)
risk_dba_address_stats["confidence"] = (
    risk_dba_address_stats["mode_count"] /
    risk_dba_address_stats["total_records"]
)

idx = pd.MultiIndex.from_frame(food_test[["DBA Name", "Address"]])
prediction = pd.Series(
    risk_dba_address_stats["predicted"].reindex(idx).to_numpy(),
    index=food_test.index
)
confidence = pd.Series(
    risk_dba_address_stats["confidence"].reindex(idx).to_numpy(),
    index=food_test.index
)

mask = (
    food_test["Risk Imputed"].isna()
    & prediction.notna()
    & (confidence >= 0.90)
)

food_test.loc[mask, "Risk Imputed"] = prediction[mask]
food_test.loc[mask, "Risk Imputation Method"] = "DBA Name + Address"

food_test["Risk Final"] = food_test["Risk Imputed"]

print(food_test["Risk Imputation Method"].value_counts())
print("Remaining missing:", food_test["Risk Final"].isna().sum())

## 4. ZIP

ZIP is a key field for merging Food Inspections with the 311 dataset, so recovering missing ZIP values is important.

Imputation order:
1. **Exact Address match**
2. **Exact Latitude + Longitude match**
3. **Nearest coordinate within 100 meters**

These rules use existing records from the dataset rather than assigning a general ZIP value.


In [ ]:
food_test["Zip Imputed"] = food_test["Zip"]
food_test["Zip Imputation Method"] = np.where(
    food_test["Zip"].notna(),
    "Original",
    "Still Missing"
)

# ---------- 1) Exact Address ----------
address_zip_stats = (
    food_test.dropna(subset=["Address", "Zip"])
    .groupby("Address")["Zip"]
    .agg(
        predicted=lambda x: x.mode().iloc[0],
        total_records="count",
        mode_count=lambda x: x.value_counts().iloc[0]
    )
)
address_zip_stats["confidence"] = (
    address_zip_stats["mode_count"] / address_zip_stats["total_records"]
)

prediction = food_test["Address"].map(address_zip_stats["predicted"])
confidence = food_test["Address"].map(address_zip_stats["confidence"])

mask = (
    food_test["Zip Imputed"].isna()
    & prediction.notna()
    & (confidence >= 0.90)
)

food_test.loc[mask, "Zip Imputed"] = prediction[mask]
food_test.loc[mask, "Zip Imputation Method"] = "Address"

# ---------- 2) Exact coordinates ----------
coord_zip_stats = (
    food_test.dropna(subset=["Latitude", "Longitude", "Zip"])
    .groupby(["Latitude", "Longitude"])["Zip"]
    .agg(
        predicted=lambda x: x.mode().iloc[0],
        total_records="count",
        mode_count=lambda x: x.value_counts().iloc[0]
    )
)
coord_zip_stats["confidence"] = (
    coord_zip_stats["mode_count"] / coord_zip_stats["total_records"]
)

coord_index = pd.MultiIndex.from_frame(food_test[["Latitude", "Longitude"]])
prediction = pd.Series(
    coord_zip_stats["predicted"].reindex(coord_index).to_numpy(),
    index=food_test.index
)
confidence = pd.Series(
    coord_zip_stats["confidence"].reindex(coord_index).to_numpy(),
    index=food_test.index
)

mask = (
    food_test["Zip Imputed"].isna()
    & prediction.notna()
    & (confidence >= 0.90)
)

food_test.loc[mask, "Zip Imputed"] = prediction[mask]
food_test.loc[mask, "Zip Imputation Method"] = "Latitude + Longitude"

# ---------- 3) Nearest coordinate within 100m ----------
known_geo = food_test.dropna(
    subset=["Latitude", "Longitude", "Zip"]
).copy()

knn = KNeighborsClassifier(n_neighbors=1)
knn.fit(
    known_geo[["Latitude", "Longitude"]],
    known_geo["Zip"]
)

remaining_mask = (
    food_test["Zip Imputed"].isna()
    & food_test["Latitude"].notna()
    & food_test["Longitude"].notna()
)

if remaining_mask.any():
    X_missing = food_test.loc[
        remaining_mask,
        ["Latitude", "Longitude"]
    ]

    predicted_zip = knn.predict(X_missing)
    distances, _ = knn.kneighbors(X_missing)

    # Approximate conversion: 1 degree latitude ≈ 111 km
    approx_distance_km = distances[:, 0] * 111

    accepted_index = X_missing.index[approx_distance_km <= 0.1]

    predicted_series = pd.Series(
        predicted_zip,
        index=X_missing.index
    )

    food_test.loc[
        accepted_index,
        "Zip Imputed"
    ] = predicted_series.loc[accepted_index]

    food_test.loc[
        accepted_index,
        "Zip Imputation Method"
    ] = "Nearest Coordinate"

print(food_test["Zip Imputation Method"].value_counts())
print("Remaining missing ZIP:", food_test["Zip Imputed"].isna().sum())

## 5. City

The original City field contains capitalization differences, repeated text, and obvious typos such as `Chicago`, `chicago`, `CCHICAGO`, `CHICAGOO`, and `CHICAGOCHICAGO`.

We first correct clear formatting and typing problems, then use the cleaned ZIP field to fill missing City values. ZIP-to-City mappings are used only when historical consistency is at least **90%**.


In [ ]:
# Basic standardization
food_test["City Clean"] = (
    food_test["City"]
    .astype("string")
    .str.strip()
    .str.upper()
    .str.replace(r"\s+", " ", regex=True)
    .str.replace(r"\.$", "", regex=True)
)

# Only corrections that were manually reviewed as clear errors
city_corrections = {
    "CCHICAGO": "CHICAGO",
    "CHICAGOO": "CHICAGO",
    "CHICAGOCHICAGO": "CHICAGO",
    "312CHICAGO": "CHICAGO",
    "CHCHICAGO": "CHICAGO",
    "CHICAGOI": "CHICAGO",
    "CHICAGOC": "CHICAGO",
    "CHCICAGO": "CHICAGO",
    "INACTIVE": "CHICAGO",
    "CHICAGOBEDFORD PARK": "CHICAGO",
    "CH": "CHICAGO",
    "CHARLES A HAYES": "CHICAGO",
    "MERRIVILLE": "MERRILLVILLE",
    "OOLYMPIA FIELDS": "OLYMPIA FIELDS",
    "NILES NILES": "NILES"
}

food_test["City Clean"] = food_test["City Clean"].replace(city_corrections)

# ZIP -> City lookup after City cleaning
city_zip_stats = (
    food_test.dropna(subset=["Zip Imputed", "City Clean"])
    .groupby("Zip Imputed")["City Clean"]
    .agg(
        predicted=lambda x: x.mode().iloc[0],
        total_records="count",
        mode_count=lambda x: x.value_counts().iloc[0]
    )
)
city_zip_stats["confidence"] = (
    city_zip_stats["mode_count"] / city_zip_stats["total_records"]
)

food_test["City Final"] = food_test["City Clean"]
food_test["City Imputation Method"] = np.where(
    food_test["City Clean"].notna(),
    "Original",
    "Still Missing"
)

prediction = food_test["Zip Imputed"].map(city_zip_stats["predicted"])
confidence = food_test["Zip Imputed"].map(city_zip_stats["confidence"])

mask = (
    food_test["City Final"].isna()
    & prediction.notna()
    & (confidence >= 0.90)
)

food_test.loc[mask, "City Final"] = prediction[mask]
food_test.loc[mask, "City Imputation Method"] = "ZIP"

print(food_test["City Imputation Method"].value_counts())
print("Remaining missing City:", food_test["City Final"].isna().sum())

## 6. State

The missing State records are associated with cleaned City values of `CHICAGO`. Therefore, missing State is filled with `IL` only when `City Final == "CHICAGO"`.

Existing non-Illinois values such as IN, CA, WI, DC, CO, and NY are preserved because their City and ZIP information supports those states.


In [ ]:
food_test["State Final"] = food_test["State"]
food_test["State Imputation Method"] = np.where(
    food_test["State"].notna(),
    "Original",
    "Still Missing"
)

mask = (
    food_test["State Final"].isna()
    & (food_test["City Final"] == "CHICAGO")
)

food_test.loc[mask, "State Final"] = "IL"
food_test.loc[mask, "State Imputation Method"] = "City"

print(food_test["State Imputation Method"].value_counts())
print("Remaining missing State:", food_test["State Final"].isna().sum())

## 7. Coordinates, Results, Inspection Type, and Violations

- **Latitude / Longitude:** Missing coordinates are kept as missing when there is no reliable exact-address reference. Using a ZIP centroid would create artificial precision.
- **Results:** No missing values; the seven result categories are retained.
- **Inspection Type:** The single missing value can be removed or handled only when that field is used in analysis.
- **Violations:** Missing values are not automatically errors. Some inspection outcomes can legitimately have no recorded violations, so these rows are retained.


In [ ]:
# Remove the single row with missing Inspection Type
food_test = food_test.dropna(subset=["Inspection Type"]).copy()

food_test["Inspection Type Clean"] = (
    food_test["Inspection Type"]
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
)

# Keep a flag for spatial analyses; do not impute fake coordinates
food_test["Coordinates Missing"] = (
    food_test["Latitude"].isna()
    | food_test["Longitude"].isna()
)

print("Results missing:", food_test["Results"].isna().sum())
print("Violations missing:", food_test["Violations"].isna().sum())
print("Coordinate pairs missing:", food_test["Coordinates Missing"].sum())

## 8. Build Final Food Dataset

Only the final analysis fields are retained. Temporary helper columns created during cleaning are removed so the output remains easy to interpret.


In [ ]:
food_clean = food_test[
    [
        "Inspection ID",
        "DBA Name",
        "AKA Name",
        "License #",
        "Facility Type Final",
        "Risk Final",
        "Address",
        "City Final",
        "State Final",
        "Zip Imputed",
        "Inspection Date",
        "Inspection Type Clean",
        "Results",
        "Violations",
        "Latitude",
        "Longitude",
        "Location",
        "Year"
    ]
].copy()

food_clean = food_clean.rename(columns={
    "Facility Type Final": "Facility Type",
    "Risk Final": "Risk",
    "City Final": "City",
    "State Final": "State",
    "Zip Imputed": "Zip",
    "Inspection Type Clean": "Inspection Type"
})

# Final data types
food_clean["Inspection Date"] = pd.to_datetime(
    food_clean["Inspection Date"],
    format="mixed",
    errors="coerce"
)

food_clean["Zip"] = (
    pd.to_numeric(food_clean["Zip"], errors="coerce")
    .astype("Int64")
    .astype("string")
)

food_clean["License #"] = (
    pd.to_numeric(food_clean["License #"], errors="coerce")
    .astype("Int64")
    .astype("string")
)

text_cols = [
    "DBA Name", "AKA Name", "Facility Type", "Risk",
    "Address", "City", "State", "Inspection Type",
    "Results", "Violations"
]

for col in text_cols:
    food_clean[col] = food_clean[col].astype("string")

print("Food final shape:", food_clean.shape)
print("\nMissing values:")
print(food_clean.isna().sum().sort_values(ascending=False))
print("\nDuplicate Inspection IDs:", food_clean["Inspection ID"].duplicated().sum())

In [ ]:
food_clean.to_csv(
    "/Users/michael818/Desktop/BA780_dataCleaning/Food_Inspections_Cleaned.csv",
    index=False
)

# Part B — 311 Sanitation

This dataset combines the historical and current 311 sanitation records. The main analytical purpose is to calculate sanitation complaint counts by **ZIP Code + Year**.

Cleaning therefore focuses on:
1. Correctly parsing historical and current date formats.
2. Removing exact duplicate rows.
3. Recovering missing ZIP values when geographic evidence is available.
4. Keeping other nonessential missing fields when they do not prevent the main analysis.


## 9. Parse Dates Correctly

Historical and current 311 records use different date string formats, so `format="mixed"` is used. Without mixed-format parsing, many current records can incorrectly become `NaT`, which would remove recent years from time-based analysis.


In [ ]:
sanitation_test["Created Date"] = pd.to_datetime(
    sanitation_test["Created Date"],
    format="mixed",
    errors="coerce"
)

sanitation_test["Closed Date"] = pd.to_datetime(
    sanitation_test["Closed Date"],
    format="mixed",
    errors="coerce"
)

print("Missing Created Date:", sanitation_test["Created Date"].isna().sum())
print(
    "Date range:",
    sanitation_test["Created Date"].min(),
    "to",
    sanitation_test["Created Date"].max()
)

## 10. Remove Exact Duplicates

Only rows that are identical across all 13 columns are removed. We do **not** deduplicate using `Request Number` alone because the same request number can appear in records with different information.


In [ ]:
before = len(sanitation_test)
sanitation_test = sanitation_test.drop_duplicates().copy()

print("Exact duplicates removed:", before - len(sanitation_test))
print("Remaining exact duplicates:", sanitation_test.duplicated().sum())

## 11. Impute Missing ZIP from Coordinates

ZIP is required for the planned `ZIP Code + Year` aggregation. Dropping all missing-ZIP rows would remove many recent records.

Most missing-ZIP records have Latitude and Longitude, so the nearest known geographic record is used to infer ZIP. This method was validated on 50,000 known records and achieved **99.31% accuracy**.

After coordinate-based imputation, the small number of rows that still have no ZIP and insufficient geographic information are removed because they cannot be used in the ZIP-level analysis.


In [ ]:
# Known ZIP + coordinate records become the geographic reference
reference = sanitation_test[
    sanitation_test["ZIP Code"].notna()
    & sanitation_test["Latitude"].notna()
    & sanitation_test["Longitude"].notna()
].copy()

ref_coords = np.radians(
    reference[["Latitude", "Longitude"]].to_numpy()
)

tree = BallTree(ref_coords, metric="haversine")

# Only impute rows where ZIP is missing but coordinates are available
missing_zip_mask = (
    sanitation_test["ZIP Code"].isna()
    & sanitation_test["Latitude"].notna()
    & sanitation_test["Longitude"].notna()
)

missing_coords = np.radians(
    sanitation_test.loc[
        missing_zip_mask,
        ["Latitude", "Longitude"]
    ].to_numpy()
)

if len(missing_coords) > 0:
    distances, indices = tree.query(missing_coords, k=1)

    sanitation_test.loc[
        missing_zip_mask,
        "ZIP Code"
    ] = reference.iloc[
        indices.flatten()
    ]["ZIP Code"].to_numpy()

print("ZIPs imputed:", missing_zip_mask.sum())
print("ZIPs still missing:", sanitation_test["ZIP Code"].isna().sum())

# Remaining rows have no reliable geographic information for the ZIP+Year analysis
before = len(sanitation_test)

sanitation_test = sanitation_test[
    sanitation_test["ZIP Code"].notna()
].copy()

print("Rows removed because ZIP could not be recovered:", before - len(sanitation_test))

## 12. Final 311 Data Types

- **ZIP Code** is an identifier, so it is stored as a string rather than a numeric measurement.
- **Ward / Police District / Community Area** are integer-like identifiers and remain nullable.
- **Closed Date** may legitimately be missing for requests that were not closed, so those rows are retained.
- Remaining coordinate or address missing values do not prevent the main ZIP + Year analysis.


In [ ]:
sanitation_test["ZIP Code"] = (
    pd.to_numeric(sanitation_test["ZIP Code"], errors="coerce")
    .astype("Int64")
    .astype("string")
)

for col in ["Ward", "Police District", "Community Area"]:
    sanitation_test[col] = (
        pd.to_numeric(sanitation_test[col], errors="coerce")
        .astype("Int64")
    )

for col in ["Latitude", "Longitude"]:
    sanitation_test[col] = pd.to_numeric(
        sanitation_test[col],
        errors="coerce"
    )

sanitation_test["Year"] = sanitation_test["Year"].astype("Int64")

print("311 final shape:", sanitation_test.shape)
print("\nMissing values:")
print(sanitation_test.isna().sum().sort_values(ascending=False))
print("\nDate range:")
print(
    sanitation_test["Created Date"].min(),
    "to",
    sanitation_test["Created Date"].max()
)
print("\nExact duplicates:", sanitation_test.duplicated().sum())

In [ ]:
sanitation_test.to_csv(
    "/Users/michael818/Desktop/BA780_dataCleaning/311_Sanitation_Cleaned_2010_2026.csv",
    index=False
)

# Final Notes for Analysis

Both cleaned datasets are now ready for analysis.

### Food Inspections
- Rows are not removed simply because `Violations`, `Facility Type`, or coordinates are missing.
- Missing values should be dropped only within an analysis that specifically requires that field.

### 311 Sanitation
- The main geographic key, `ZIP Code`, is complete after cleaning.
- Remaining missing values in fields such as `Closed Date`, Ward, Police District, Community Area, coordinates, or Street Address do not block the planned ZIP-level analysis.
- The next step is to aggregate sanitation complaints by **ZIP Code + Year** and merge them with Food Inspections.
